# Chapter 7 — Episodic Memory

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting the backbone LLM of your agent

These notebooks run on **Ollama by default**, the setup the book teaches.
If you do nothing, nothing changes: `make_llm()` starts a local Ollama
service when one isn't already running.

To use OpenAI or Anthropic instead:

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you off the Ollama path.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama regardless
of these settings.

If you opted in but forgot to export the key, you will be prompted for it.
That is the safe path on hosted kernels, and it keeps the key out of the
saved notebook. Setting `OLLAMA_API_KEY` alone routes Ollama to Ollama
Cloud.

**Caveat:** the examples are tuned for qwen3. Output on gpt-5 or Claude
will differ from what is printed in the book, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [2]:
from llm_agents_from_scratch.notebook_utils import make_llm

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Examples

### Example 1: Constructing an Episode and Printing Format Modes

In [3]:
from llm_agents_from_scratch.data_structures import Task, TaskResult
from llm_agents_from_scratch.data_structures.memory import Episode, EpisodeFormatMode

task = Task(instruction="Compute the Hailstone sequence for 6.")
result = TaskResult(task_id=task.id_, content="6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1")
episode = Episode(
    task=task,
    rollout="mock rollout",
    result=result,
    metadata={"reflection": "Always use the hailstone tool."},
)

print("=== XML (default) ===")
print(episode.format(mode=EpisodeFormatMode.XML))

print()
print("=== CONCAT ===")
print(episode.format(mode=EpisodeFormatMode.CONCAT))

=== XML (default) ===
  <episode>
    <task>Compute the Hailstone sequence for 6.</task>
    <result>6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1</result>
    <reflection>Always use the hailstone tool.</reflection>
    <completed_at>2026-10-11 11:35:16</completed_at>
  </episode>

=== CONCAT ===
task: Compute the Hailstone sequence for 6.
result: 6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1
reflection: Always use the hailstone tool.
completed_at: 2026-10-11 11:35:16


### Example 2a: Writing Episodes to a QdrantMemoryStore

In [4]:
import warnings

from llm_agents_from_scratch.data_structures import Task, TaskResult
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.memory_stores.qdrant import QdrantMemoryStore

warnings.filterwarnings("ignore", message="Payload indexes have no effect")

store = QdrantMemoryStore(max_results=1)

task_1 = Task(instruction="Compute the Hailstone sequence for 6.")
task_2 = Task(instruction="Compute the Hailstone sequence for 8.")

episodes = [
    Episode(
        task=task_1,
        rollout="mock rollout",
        result=TaskResult(
            task_id=task_1.id_,
            content="6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1",
        ),
        metadata={"steps": "8"},
    ),
    Episode(
        task=task_2,
        rollout="mock rollout",
        result=TaskResult(
            task_id=task_2.id_,
            content="8 → 4 → 2 → 1",
        ),
        metadata={"steps": "3"},
    ),
]

for ep in episodes:
    await store.write(ep)
    print(f"count: {await store.count()}")

count: 1
count: 2


### Example 2b: Recalling from a QdrantMemoryStore

In [5]:
results = await store.recall("Hailstone sequence starting from 6")
print(
    f"recall() returned {len(results)} episode(s)"
    " -- query matched by similarity (RecallMode.SEARCH)\n"
)
print(results[0].format())

recall() returned 1 episode(s) -- query matched by similarity (RecallMode.SEARCH)

  <episode>
    <task>Compute the Hailstone sequence for 6.</task>
    <result>6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1</result>
    <steps>8</steps>
    <completed_at>2026-10-11 11:35:16</completed_at>
  </episode>


### Example 3: Memory with a metadata_fn

In [6]:
import warnings

from llm_agents_from_scratch.data_structures import Task, TaskResult
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.memory import Memory
from llm_agents_from_scratch.memory_stores.qdrant import QdrantMemoryStore

warnings.filterwarnings("ignore", message="Payload indexes have no effect")

task = Task(instruction="Compute the Hailstone sequence for 6.")
episode = Episode(
    task=task,
    rollout="mock rollout",
    result=TaskResult(
        task_id=task.id_,
        content="6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1",
    ),
)


def step_counter_fn(ep: Episode) -> str:
    return str(len(ep.result.content.split("→")))


store = QdrantMemoryStore(max_results=1)
memory = Memory(store=store, metadata_fns={"steps": step_counter_fn})

await memory.record(episode)

# record() works on a deep copy — the original episode is not mutated
print("Original episode:")
print(episode)

recalled = await memory.recall(Task(instruction="Compute the Hailstone sequence for 3."))
print("\nRecalled episode (enriched by step_counter_fn at write time):")
print(recalled)

Original episode:
  <episode>
    <task>Compute the Hailstone sequence for 6.</task>
    <result>6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1</result>
    <completed_at>2026-10-11 11:35:17</completed_at>
  </episode>

Recalled episode (enriched by step_counter_fn at write time):
  <episode>
    <task>Compute the Hailstone sequence for 6.</task>
    <result>6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1</result>
    <steps>9</steps>
    <completed_at>2026-10-11 11:35:17</completed_at>
  </episode>


### Example 4: LLMAgent with similarity_memory()

In [7]:
import logging
import warnings

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory.recipes import similarity_memory
from llm_agents_from_scratch.tools import SimpleFunctionTool

warnings.filterwarnings("ignore", message="Payload indexes have no effect")
enable_console_logging(logging.INFO)


def next_number(n: int) -> int:
    """Returns the next number in the stop-at-one sequence."""
    return n // 2 if n % 2 == 0 else 3 * n + 1

next_number_tool = SimpleFunctionTool(func=next_number)
llm = make_llm()
memory = similarity_memory(max_results=1)
agent = LLMAgent(llm=llm, tools=[next_number_tool], memories=[memory])

✓ Using Ollama Cloud (gemma4:31b)


In [8]:
agent.llm.model

'gemma4:31b'

In [9]:
# Task 1: no memory yet — the agent computes the sequence via tools and
# similarity_memory records the episode in the Qdrant vector store.
result_1 = await agent.run(
    Task(
        instruction=(
            "Compute the stop-at-one sequence for 4"
        )
    ),
)
print(result_1.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the stop-at-one sequence for 4
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the stop-at-one sequence for 4
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 2
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "cad55d17-8bc6-4f98-b9dc-4f9f2451b08d",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Call the next_number tool with n=2 to continue computing the sequence.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Call the next_number tool with n=2 to continue computing the sequence.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 1
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: The next n

In [10]:
# Inspect what will be injected into the system prompt as memory for Task 2.
# This is the same call the agent makes internally via memory.recall() before
# its first step — surfacing it here makes the memory context visible.
task_2 = Task(
    instruction=(
        "What is the stop-at-one sequence for 4?"
    )
)
print("Memory recalled for Task 2:\n")
print(await memory.recall(task_2))

Memory recalled for Task 2:

  <episode>
    <task>Compute the stop-at-one sequence for 4</task>
    <result>The next number in the sequence after 2 is 1. Since we have reached 1, the stop-at-one sequence for 4 is: 4, 2, 1.</result>
    <completed_at>2026-10-11 11:35:20</completed_at>
  </episode>


In [11]:
# Task 2: the episode from Task 1 is recalled by similarity search and
# injected into the system prompt — the agent answers directly from memory.
result_2 = await agent.run(
    task_2,
)
print(result_2.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: What is the stop-at-one sequence for 4?
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: What is the stop-at-one sequence for 4?
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: <episode>
    <task>Compute the stop-at-one sequence for 4</task>
    <result>The next number in the sequence after 2 is 1. Since we ha...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      No new step required.
INFO (llm_agents_fs.LLMAgent) :      🏁 Task completed: <episode>
    <task>Compute the stop-at-one sequence for 4</task>
    <result>The next number in the sequence after 2 is 1. Since we...[TRUNCATED]
<episode>
    <task>Compute the stop-at-one sequence for 4</task>
    <result>The next number in the sequence after 2 is 1. Since we have reached 1, the stop-at-one sequence for 4 is: 4, 2, 1.</result>
    <completed_at>2026-10-11 11:35:20</completed_at>
</episode>

The stop-at-one sequence for 4 is: 4, 2, 1.
